# Stored execution evidence — RANZCR CLiP (inference)

This public evidence copy preserves every textual output cell supplied in `agent_final_submission.zip`.
The original notebook SHA-256 is `32ab7be134c62c0fdada065bbe0cbbc3566d2893c8055df7d37d3ed016e244f2`. Embedded binary rich media is removed so
that competition data and generated images are not redistributed. The repository setup cell
now targets `Isso-W/Jiaozi` at `main`; stored metrics and submission receipts remain historical
outputs and are not presented as a fresh rerun of the current branch.


# Jiaozi Agent — RANZCR CLiP Kaggle inference

This is the Kaggle-side submission Notebook. Attach both inputs before running:

1. Competition data: `ranzcr-clip-catheter-line-classification`
2. The private Kaggle Dataset created from `ranzcr_dinov3_assets.zip`

Select a compatible GPU such as **T4**, keep Internet off, and use **Save Version → Save & Run All**. The final cell restores the exact trained multi-label model and creates `/kaggle/working/submission.csv`. It does not retrain the model.


In [1]:
# Locate competition data and the packaged Jiaozi checkpoint.

import json
import os
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from transformers import AutoConfig, AutoModel

KAGGLE_INPUT = Path("/kaggle/input")
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)

if not KAGGLE_INPUT.exists():
    raise RuntimeError("Run this Notebook in Kaggle with attached inputs.")

sample_candidates = sorted(KAGGLE_INPUT.rglob("sample_submission.csv"))
sample_candidates = [
    path for path in sample_candidates
    if "ranzcr-clip-catheter-line-classification" in str(path.parent).lower()
]
if not sample_candidates:
    raise FileNotFoundError(
        "Competition sample_submission.csv was not found. Attach the "
        "ranzcr-clip-catheter-line-classification competition input."
    )

SAMPLE_PATH = sample_candidates[0]
COMPETITION_ROOT = next(
    parent for parent in [SAMPLE_PATH.parent, *SAMPLE_PATH.parents]
    if parent.name == "ranzcr-clip-catheter-line-classification"
)

checkpoint_candidates = sorted(KAGGLE_INPUT.rglob("best_model.pt"))
if not checkpoint_candidates:
    asset_zips = sorted(
        path for path in KAGGLE_INPUT.rglob("*.zip")
        if "ranzcr_dinov3_assets" in path.name.lower()
    )
    if not asset_zips:
        raise FileNotFoundError(
            "best_model.pt or ranzcr_dinov3_assets.zip was not found. "
            "Attach the private RANZCR model asset Dataset."
        )
    extracted_assets = WORKING_DIR / "restored_ranzcr_assets"
    extracted_assets.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(asset_zips[0], "r") as archive:
        archive.extractall(extracted_assets)
    checkpoint_candidates = sorted(extracted_assets.rglob("best_model.pt"))

CHECKPOINT_PATH = checkpoint_candidates[0]
ASSET_DIR = CHECKPOINT_PATH.parent
CONFIG_PATH = ASSET_DIR / "config.json"
MANIFEST_PATH = ASSET_DIR / "asset_manifest.json"
if not CONFIG_PATH.is_file() or not MANIFEST_PATH.is_file():
    raise FileNotFoundError("The model asset must include config.json and asset_manifest.json")

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
sample = pd.read_csv(SAMPLE_PATH)
id_column = manifest["id_column"]
label_columns = list(manifest["label_columns"])
expected_columns = [id_column, *label_columns]
if sample.columns.tolist() != expected_columns:
    raise ValueError(
        f"Submission columns do not match the checkpoint contract. "
        f"Expected {expected_columns}, received {sample.columns.tolist()}"
    )

print("torch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
print("competition root:", COMPETITION_ROOT)
print("sample submission:", SAMPLE_PATH)
print("checkpoint:", CHECKPOINT_PATH)
print("test rows:", len(sample))
print("target labels:", len(label_columns))
print("best epoch:", manifest.get("best_epoch"))
print("validation mean ROC-AUC:", manifest.get("best_metric"))


torch: 2.10.0+cu128
GPU available: True
competition root: /kaggle/input/competitions/ranzcr-clip-catheter-line-classification
sample submission: /kaggle/input/competitions/ranzcr-clip-catheter-line-classification/sample_submission.csv
checkpoint: /kaggle/input/datasets/baron66/jiaozi-ranzcr-dinov3-assets-v2/best_model.pt
test rows: 3582
target labels: 11
best epoch: 16
validation mean ROC-AUC: 0.8577258824877178


In [2]:
# Rebuild the DINOv3 multi-label model and restore every tensor.

def hidden_size(backbone):
    config = backbone.config
    for name in ("hidden_size", "embed_dim", "hidden_sizes"):
        value = getattr(config, name, None)
        if isinstance(value, int):
            return value
        if isinstance(value, (list, tuple)) and value:
            return int(value[-1])
    raise ValueError(
        f"Could not determine hidden size from {type(config).__name__}"
    )


def pooled_features(outputs):
    pooled = getattr(outputs, "pooler_output", None)
    if pooled is not None:
        return pooled

    last = getattr(outputs, "last_hidden_state", None)

    if last is None and isinstance(outputs, (tuple, list)):
        last = outputs[0]

    if last is None:
        raise ValueError("Backbone output has no usable feature tensor")

    if last.ndim == 4:
        return last.mean(dim=(-2, -1))

    if last.ndim == 3:
        return last[:, 0]

    return last


class MultiLabelModel(nn.Module):
    def __init__(self, backbone, num_labels, dropout=0.1):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size(backbone), num_labels),
        )

    def forward(self, pixel_values):
        outputs = self.backbone(pixel_values=pixel_values)
        return self.classifier(pooled_features(outputs))


# Load the epoch-16 best checkpoint.
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

if list(checkpoint["label_columns"]) != label_columns:
    raise ValueError(
        "Checkpoint labels do not match asset_manifest.json"
    )

# Rebuild DINOv3 from the offline configuration.
config = AutoConfig.from_pretrained(
    ASSET_DIR,
    local_files_only=True,
)
backbone = AutoModel.from_config(config)
model = MultiLabelModel(
    backbone,
    len(label_columns),
)

# Compatibility mapping:
# Transformers 5.12.1:
# backbone.model.layer...
#
# Kaggle Transformers 5.0.0:
# backbone.layer...
saved_state = checkpoint["model_state_dict"]
compatible_state = {}
renamed_keys = []

for saved_key, tensor in saved_state.items():
    if saved_key.startswith("backbone.model."):
        current_key = (
            "backbone."
            + saved_key[len("backbone.model."):]
        )
        renamed_keys.append((saved_key, current_key))
    else:
        current_key = saved_key

    if current_key in compatible_state:
        raise RuntimeError(
            "Duplicate key after compatibility mapping: "
            f"{current_key}"
        )

    compatible_state[current_key] = tensor


# Strictly verify all names and tensor shapes before loading.
current_state = model.state_dict()

missing_keys = sorted(
    set(current_state).difference(compatible_state)
)
unexpected_keys = sorted(
    set(compatible_state).difference(current_state)
)

shape_mismatches = [
    (
        key,
        tuple(compatible_state[key].shape),
        tuple(current_state[key].shape),
    )
    for key in sorted(
        set(current_state).intersection(compatible_state)
    )
    if compatible_state[key].shape != current_state[key].shape
]

if missing_keys or unexpected_keys or shape_mismatches:
    raise RuntimeError(
        "Checkpoint compatibility validation failed:\n"
        f"missing={missing_keys[:10]}\n"
        f"unexpected={unexpected_keys[:10]}\n"
        f"shape mismatches={shape_mismatches[:10]}"
    )


# Load every tensor strictly.
load_result = model.load_state_dict(
    compatible_state,
    strict=True,
)

if load_result.missing_keys or load_result.unexpected_keys:
    raise RuntimeError(
        "Checkpoint mismatch after loading:\n"
        f"missing={load_result.missing_keys}\n"
        f"unexpected={load_result.unexpected_keys}"
    )


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
model = model.to(device).eval()

print("model tensors restored:", len(saved_state))
print("DINOv3 compatibility keys renamed:", len(renamed_keys))
print("strict checkpoint validation: PASSED")
print(
    "classification head:",
    tuple(model.classifier[1].weight.shape),
)
print("device:", device)


model tensors restored: 213
DINOv3 compatibility keys renamed: 204
strict checkpoint validation: PASSED
classification head: (11, 768)
device: cuda


In [3]:
# Run RANZCR test inference and write /kaggle/working/submission.csv.

test_ids = sample[id_column].astype(str).tolist()
first_image_name = f"{test_ids[0]}.jpg"
first_matches = [
    path for path in COMPETITION_ROOT.rglob(first_image_name)
    if path.is_file()
]

if not first_matches:
    test_zip_candidates = sorted(COMPETITION_ROOT.rglob("test.zip"))
    if not test_zip_candidates:
        test_zip_candidates = sorted(
            path for path in COMPETITION_ROOT.rglob("*.zip")
            if "test" in path.name.lower()
        )
    if not test_zip_candidates:
        raise FileNotFoundError(
            f"Neither {first_image_name} nor a test archive was found under {COMPETITION_ROOT}"
        )
    extracted_test_root = WORKING_DIR / "ranzcr_test_images"
    extracted_test_root.mkdir(parents=True, exist_ok=True)
    print("Extracting Kaggle test archive:", test_zip_candidates[0])
    with zipfile.ZipFile(test_zip_candidates[0], "r") as archive:
        archive.extractall(extracted_test_root)
    first_matches = [
        path for path in extracted_test_root.rglob(first_image_name)
        if path.is_file()
    ]

if not first_matches:
    raise FileNotFoundError(f"First test image not found: {first_image_name}")

candidate_dirs = sorted({path.parent.resolve() for path in first_matches})
test_image_dir = max(
    candidate_dirs,
    key=lambda directory: sum(
        (directory / f"{image_id}.jpg").is_file()
        for image_id in test_ids[:100]
    ),
)

missing = [
    image_id for image_id in test_ids
    if not (test_image_dir / f"{image_id}.jpg").is_file()
]
if missing:
    raise FileNotFoundError(
        f"Missing {len(missing)} test images; first IDs: {missing[:10]}"
    )

image_size = int(checkpoint["image_size"])
test_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=manifest["normalization_mean"],
        std=manifest["normalization_std"],
    ),
])


class RanzcrTestDataset(Dataset):
    def __init__(self, ids, image_dir):
        self.ids = ids
        self.image_dir = image_dir

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        image_id = self.ids[index]
        with Image.open(self.image_dir / f"{image_id}.jpg") as image:
            pixel_values = test_transform(image.convert("RGB"))
        return image_id, pixel_values


loader = DataLoader(
    RanzcrTestDataset(test_ids, test_image_dir),
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

prediction_ids = []
prediction_values = []
with torch.inference_mode():
    for batch_index, (batch_ids, pixel_values) in enumerate(loader, start=1):
        pixel_values = pixel_values.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
            logits = model(pixel_values)
            probabilities = torch.sigmoid(logits)
        prediction_ids.extend(list(batch_ids))
        prediction_values.append(probabilities.float().cpu().numpy())
        if batch_index == 1 or batch_index % 10 == 0 or batch_index == len(loader):
            print(f"[infer] batch {batch_index}/{len(loader)}")

prediction_matrix = np.concatenate(prediction_values, axis=0)
predictions = pd.DataFrame(prediction_matrix, columns=label_columns)
predictions.insert(0, id_column, prediction_ids)
submission = sample[[id_column]].merge(predictions, on=id_column, how="left")
values = submission[label_columns].to_numpy(dtype=float)

assert submission.columns.tolist() == expected_columns
assert submission.shape == sample.shape
assert submission[id_column].astype(str).equals(sample[id_column].astype(str))
assert np.isfinite(values).all()
assert ((values >= 0.0) & (values <= 1.0)).all()
assert np.std(values) > 0.0

SUBMISSION_PATH = WORKING_DIR / "submission.csv"
submission.to_csv(SUBMISSION_PATH, index=False)

print("\n=== KAGGLE SUBMISSION READY ===")
print("file:", SUBMISSION_PATH)
print("shape:", submission.shape)
print("probability range:", float(values.min()), "to", float(values.max()))
print("probability std:", float(values.std()))
display(submission.head())


[infer] batch 1/112
[infer] batch 10/112
[infer] batch 20/112
[infer] batch 30/112
[infer] batch 40/112
[infer] batch 50/112
[infer] batch 60/112
[infer] batch 70/112
[infer] batch 80/112
[infer] batch 90/112
[infer] batch 100/112
[infer] batch 110/112
[infer] batch 112/112

=== KAGGLE SUBMISSION READY ===
file: /kaggle/working/submission.csv
shape: (3582, 12)
probability range: 5.960464477539063e-08 to 1.0
probability std: 0.2901637591789588


,StudyInstanceUID,ETT - Abnormal,ETT - Borderline,ETT - Normal,NGT - Abnormal,NGT - Borderline,NGT - Incompletely Imaged,NGT - Normal,CVC - Abnormal,CVC - Borderline,CVC - Normal,Swan Ganz Catheter Present
0,1.2.826.0.1.3680043.8.498.46923145579096002617...,0.453125,0.923340,0.810059,0.094849,0.405762,0.434814,0.944336,0.232788,0.394043,0.941406,1.000000e+00
1,1.2.826.0.1.3680043.8.498.84006870182611080091...,0.000636,0.000436,0.000177,0.000296,0.002714,0.006050,0.005161,0.449951,0.416504,0.708984,1.937151e-05
2,1.2.826.0.1.3680043.8.498.12219033294413119947...,0.000035,0.000098,0.000457,0.000225,0.001192,0.000358,0.000989,0.142578,0.336182,0.840820,4.649162e-06
3,1.2.826.0.1.3680043.8.498.84994474380235968109...,0.456787,0.763184,0.249756,0.722656,0.062805,0.927734,0.119385,0.597168,0.567383,0.586914,6.587982e-03
4,1.2.826.0.1.3680043.8.498.35798987793805669662...,0.000118,0.001245,0.000521,0.000823,0.009712,0.000043,0.012825,0.174316,0.335205,0.812988,7.152557e-07
